In [ ]:
#The purpose of this notebook is to replicate the results of Ghorbani et al. results for Dmax regression using XGBoost.
#Ghorbani used a RandomForestRegressor(max_depth=12, n_estimators=45) from scikit-learn

In [35]:
#import the required libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import re
from sklearn.model_selection import RepeatedKFold,ShuffleSplit

In [24]:
#Take formula column and parse it into a dataframe of element columns with atomic percentages as values. This function should be able to handle the complex formulas in the Ghorbani dataset, including nested parentheses, brackets, and braces, as well as fractions and equal splits.
def assemble_composition_df(df, formula_column):
    element_list = [
        "Ag", "Al", "Am", "As", "Au",
        "B", "Ba", "Be", "Bi",
        "C", "Ca", "Cd", "Ce", "Co", "Cr", "Cs", "Cu",
        "Dy",
        "Er", "Eu",
        "Fe",
        "Ga", "Gd", "Ge",
        "H", "Hf", "Hg", "Ho",
        "In", "Ir",
        "K",
        "La", "Li", "Lu",
        "Mg", "Mn", "Mo",
        "N", "Na", "Nb", "Nd", "Ni", "Np",
        "O", "Os",
        "P", "Pa", "Pb", "Pd", "Pr", "Pt", "Pu",
        "Rb", "Re", "Rh", "Ru",
        "S", "Sb", "Sc", "Se", "Si", "Sm", "Sn", "Sr",
        "Ta", "Tb", "Tc", "Te", "Th", "Ti", "Tl", "Tm",
        "U",
        "V",
        "W",
        "Y", "Yb",
        "Zn", "Zr"
    ]
    
    def parse_fraction(s):
        """Parse a string that might be a fraction (e.g., '5/6') or a number."""
        if '/' in s:
            num, denom = s.split('/')
            return float(num) / float(denom)
        return float(s)
    
    def parse_element_composition(formula_str):
        """
        Parse element-number pairs from a formula string.
        Returns a dict of {element: amount}
        """
        composition = {}
        # Pattern to match element followed by optional number (including fractions)
        pattern = r'([A-Z][a-z]?)(\d+(?:\.\d+)?(?:/\d+(?:\.\d+)?)?)?'
        
        matches = re.findall(pattern, formula_str)
        for element, amount in matches:
            if element and element in element_list:
                if amount:
                    val = parse_fraction(amount)
                else:
                    val = 1.0
                composition[element] = composition.get(element, 0) + val
        
        return composition
    
    def parse_formula(formula):
        """
        Parse a complete alloy formula handling nested brackets, parentheses, and braces.
        Returns a dict of {element: atomic_percent}
        """
        composition = {}
        
        # Remove citation references like [24], [30], etc. at the end
        formula = re.sub(r'\[\d+\]$', '', formula)
        formula = re.sub(r'\[\d+\]', '', formula)
        
        # Remove spaces and commas used as separators
        formula = formula.replace(' ', '').replace(',', '')
        
        def process_innermost_group(f):
            """Find and process the innermost bracketed group."""
            pattern = r'([\(\[\{])([^\(\)\[\]\{\}]+)([\)\]\}])(\d+(?:\.\d+)?)?'
            
            match = re.search(pattern, f)
            if not match:
                return f, False
            
            open_bracket, content, close_bracket, multiplier = match.groups()
            
            # Parse the content of the group
            inner_comp = parse_element_composition(content)
            
            # Calculate the sum of inner compositions
            inner_sum = sum(inner_comp.values())
            
            # Determine the multiplier
            if multiplier:
                mult = float(multiplier)
            else:
                mult = 1.0
            
            # Determine if inner values are fractions or percentages
            # If sum is close to 1, treat as fractions; if close to 100, treat as percentages
            if inner_sum > 1.5:  # Likely percentages within the group
                # Normalize to fractions, then multiply
                inner_comp = {k: v / inner_sum for k, v in inner_comp.items()}
            
            # Apply multiplier
            expanded = {elem: amt * mult for elem, amt in inner_comp.items()}
            
            # Create replacement string
            replacement_parts = []
            for elem, amt in expanded.items():
                replacement_parts.append(f"{elem}{amt}")
            replacement = ''.join(replacement_parts)
            
            new_f = f[:match.start()] + replacement + f[match.end():]
            
            return new_f, True
        
        # Iteratively process innermost groups until none remain
        processed_formula = formula
        max_iterations = 20
        iteration = 0
        
        while iteration < max_iterations:
            processed_formula, found = process_innermost_group(processed_formula)
            if not found:
                break
            iteration += 1
        
        # Now parse the final expanded formula
        composition = parse_element_composition(processed_formula)
        
        # Handle equal split case (elements with no numbers)
        total = sum(composition.values())
        num_elements = len(composition)
        
        # Check if all elements have value 1.0 (no numbers given)
        if num_elements > 0 and all(v == 1.0 for v in composition.values()):
            equal_share = 100.0 / num_elements
            composition = {k: equal_share for k in composition}
        # If total is very small (< 2), scale up to 100
        elif total > 0 and total < 2:
            scale = 100.0 / total
            composition = {k: v * scale for k, v in composition.items()}
        
        return composition
    
    # Process all formulas
    composition_dicts = []
    for formula in df[formula_column]:
        try:
            comp = parse_formula(str(formula))
            composition_dicts.append(comp)
        except Exception as e:
            print(f"Error parsing '{formula}': {e}")
            composition_dicts.append({})
    
    # Create DataFrame with element columns
    comp_df = pd.DataFrame(composition_dicts)
    
    # Ensure all element columns exist, fill missing with 0
    for elem in element_list:
        if elem not in comp_df.columns:
            comp_df[elem] = 0.0
    
    # Reorder columns to match element_list and fill NaN with 0
    comp_df = comp_df.reindex(columns=element_list, fill_value=0.0)
    comp_df = comp_df.fillna(0.0)
    
    return comp_df

#take composition df and produce composition strings
def canonical_comp_string(df, tol=1e-9, decimals=2):
    element_cols = sorted([c for c in df.columns if c != "Composition String"])
    out = []
    for _, row in df[element_cols].iterrows():
        vals = row.astype(float).fillna(0.0).to_numpy()
        vals[vals < tol] = 0.0
        s = vals.sum()
        if s <= 0:
            out.append("")
            continue
        vals = vals / s * 100.0
        vals = np.round(vals, decimals)
        parts = [f"{el}{v:.{decimals}f}" for el, v in zip(element_cols, vals) if v > 0]
        out.append("".join(parts))
    return out


In [30]:
#load the Ghorbani dataset
raw_data = pd.read_excel(r"Data\Paper Data\Ghorbani, 2022.xlsx")
raw_data.columns

Index(['No.', 'Alloy', 'Tg', 'Tx', 'Tl', 'X1 (Trg)', 'X2 (Delta T)',
       'X3 (Alpha)', 'X4 (Beta)', 'X5 (new Beta)', 'X6 (Gamma)',
       'X7 (Gamma m)', 'X8 (Delta)', 'X9 (NULL sign)', 'X10 (Omega)',
       'X11 (Omega m)', 'X12 (Theta)', 'X13 (Xi)', 'X14 (Beta Prime)',
       'X15 (Dleta Trg)', 'X16 (Gp)', 'X17 (Gamma C)', 'Y (Dmax)'],
      dtype='object')

In [31]:
#create composition dataframe from the raw data
composition_df = assemble_composition_df(raw_data, "Alloy")

#find entries that have totals greater than 100
over_100 = composition_df.sum(axis=1) > 100
over_100_count = sum(over_100)
print(f"Number of entries with totals greater than 100: {over_100_count}")

#drop number of entries with totals greater than 100
n_before = len(composition_df)
composition_df = composition_df[~over_100]
n_after = len(composition_df)
print(f"Entries before dropping: {n_before}, Entries after dropping: {n_after}")

#use the same over 100 index to drop from the raw data
raw_data = raw_data[~over_100]

Number of entries with totals greater than 100: 26
Entries before dropping: 715, Entries after dropping: 689


In [32]:
#assemble composition strings and add to the raw data dataframe
canonical_comp_strings = canonical_comp_string(composition_df)
raw_data["Composition String"] = canonical_comp_strings


#check raw data frame for duplicate composition strings
duplicates = raw_data["Composition String"].duplicated().sum()
print(f"Number of duplicate composition strings: {duplicates}")
len_pre_dup = len(raw_data)

#replace duplicates with average values
raw_data = raw_data.groupby("Composition String").mean(numeric_only=True).reset_index()
len_post_dup = len(raw_data)
print(f"Entries before removing duplicates: {len_pre_dup}, Entries after removing duplicates: {len_post_dup}")

Number of duplicate composition strings: 28
Entries before removing duplicates: 689, Entries after removing duplicates: 661


In [33]:
raw_data.columns

Index(['Composition String', 'No.', 'Tg', 'Tx', 'Tl', 'X1 (Trg)',
       'X2 (Delta T)', 'X3 (Alpha)', 'X4 (Beta)', 'X5 (new Beta)',
       'X6 (Gamma)', 'X7 (Gamma m)', 'X8 (Delta)', 'X9 (NULL sign)',
       'X10 (Omega)', 'X11 (Omega m)', 'X12 (Theta)', 'X13 (Xi)',
       'X14 (Beta Prime)', 'X15 (Dleta Trg)', 'X16 (Gp)', 'X17 (Gamma C)',
       'Y (Dmax)'],
      dtype='object')

In [37]:
#split the raw data into features and target
X = raw_data.copy().drop(['No.', 'Composition String','Y (Dmax)'], axis=1)
y = raw_data['Y (Dmax)']

#separate into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [39]:
#replicate the random forest regression model and the cv used by Ghorbani et al. which is a repeated k-fold with 5 splits and 20 repeats
cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=123)

for X_fold, y_fold in cv.split(X_train, y_train):
    print(X_train.iloc[X_fold], y_train.iloc[y_fold])
    break

        Tg     Tx      Tl  X1 (Trg)  X2 (Delta T)  X3 (Alpha)  X4 (Beta)  \
533  358.0  378.0   664.0  0.539157          20.0    0.569277   1.595023   
552  365.0  441.0   681.0  0.535977          76.0    0.647577   1.744196   
613  684.7  722.1  1164.5  0.587941          37.4    0.620087   1.642837   
61   711.0  800.0  1186.0  0.599494          89.0    0.674536   1.724670   
430  829.0  893.0  1493.0  0.555258          64.0    0.598125   1.632459   
..     ...    ...     ...       ...           ...         ...        ...   
20   694.0  726.0  1130.0  0.614159          32.0    0.642478   1.660269   
71   405.0  456.0   730.0  0.554795          51.0    0.624658   1.680720   
106  688.0  768.0  1137.0  0.605101          80.0    0.675462   1.721380   
270  465.2  541.8   822.5  0.565593          76.6    0.658723   1.730253   
435  822.0  870.0  1481.0  0.555030          48.0    0.587441   1.613425   

     X5 (new Beta)  X6 (Gamma)  X7 (Gamma m)  X8 (Delta)  X9 (NULL sign)  \
533       1